#### **벡터화**

- 토큰화 작업을 통해서 단어들을 추출했다면 해당 단어들을 숫자형으로 변환
    - 이유는? 연산을 하기 위함
- 벡터화를 통해서 숫자형으로 데이터를 변환하고 label(정답) 데이터로 규칙을 찾아간다.

<br>

- 자연어를 이용한 예측 모델 순서
    1. 데이터 로드
    2. 문자 토큰화
    3. 벡터화
    4. 모델 학습
    5. 평가

In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from konlpy.tag import Okt

In [27]:
df = pd.read_csv('../data/ratings_train.txt', sep='\t')
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        150000 non-null  int64
 1   document  149995 non-null  str  
 2   label     150000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 3.4 MB


In [29]:
# document column에서 ' '은 결측치가 아니다. '  ', '   ' 도 마찬가지
# document column에서 strip() 함수를 사용

df['document'] = df['document'].str.strip()
df.loc[df['document'] == '', 'document'] = np.nan

In [30]:
# document의 결측치 데이터를 제외
df.dropna(inplace=True)

In [31]:
# ID 데이터가 고유한 데이터인가?
len(df['id'].unique())

149995

In [32]:
df['id'].value_counts()

id
9976970     1
3819312     1
10265843    1
9045019     1
6483659     1
           ..
6222902     1
8549745     1
9311800     1
2376369     1
9619869     1
Name: count, Length: 149995, dtype: int64

In [33]:
# 중복으로 작성되어 있는 리뷰가 존재한다면 제거
df['document'].value_counts()

document
굿                                  181
good                                92
최고                                  85
쓰레기                                 79
별로                                  66
                                  ... 
인간이 문제지.. 소는 뭔죄인가..                  1
평점이 너무 낮아서...                        1
이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?      1
청춘 영화의 최고봉.방황과 우울했던 날들의 자화상          1
한국 영화 최초로 수간하는 내용이 담긴 영화             1
Name: count, Length: 146182, dtype: int64

In [34]:
# 중복 제거
df.drop_duplicates('document', inplace=True)

In [35]:
df['document'].value_counts()

document
아 더빙.. 진짜 짜증나네요 목소리                                              1
흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나                                1
너무재밓었다그래서보는것을추천한다                                                1
교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정                                    1
사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다    1
                                                                ..
인간이 문제지.. 소는 뭔죄인가..                                              1
평점이 너무 낮아서...                                                    1
이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?                                  1
청춘 영화의 최고봉.방황과 우울했던 날들의 자화상                                      1
한국 영화 최초로 수간하는 내용이 담긴 영화                                         1
Name: count, Length: 146182, dtype: int64

In [36]:
# id 항목은 고유값, 모델 학습에서 의미가 없음 → 제거
df.drop('id', axis=1, inplace=True)

In [37]:
x = df['document']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42
)

In [38]:
y.value_counts()

label
0    73342
1    72840
Name: count, dtype: int64

In [39]:
# 토큰화 작업 → 2개의 필터링: 품사 체크, 불용어 처리
okt = Okt()

# 사용할 품사 종류
allow_pos = ['Noun', 'Verb', 'Adjective', 'KoreanParicle', 'Alpha']

# 불용어 단어 목록
stop_word = ['하다', '되다']

# 글자 수 제한
len_word = 2

# 토큰화 함수
def tokenize(input_text):
    # 결과를 리스트로 되돌려준다. → 빈 리스트 생성
    result = []

    for word, pos in okt.pos(input_text, norm=True, stem=True):
        
        # 조건1: allow_pos에 포함되어 있다면
        # 조건2: stop_word에 포함되어 있지 않다면
        # 조건3: 길이가 len_word보다 크거나 같은 경우

        if ( pos in allow_pos ) & ( word not in stop_word ) & ( len(word) >= len_word ):
            result.append(word)
    
    return result

In [40]:
# 벡터화
# CountVectorizer → 존재 여부, 단어의 개수

vectorizer = CountVectorizer(
    tokenizer=tokenize,
    lowercase=False,
    binary=True
)

X_train_vec = vectorizer.fit_transform(X_train)

c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [41]:
vectorizer.get_feature_names_out()

array(['ABBA', 'ACTION', 'ADHD', ..., '힛힛', '힝힝', '힣히히헤'],
      shape=(37722,), dtype=object)

In [42]:
X_train_vec

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 832562 stored elements and shape (116945, 37722)>

In [43]:
X_test_vec = vectorizer.transform(X_test)

In [44]:
model = LinearSVC(C = 1.0)

In [45]:
model.fit(X_train_vec, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [48]:
pred = model.predict(X_test_vec)

In [ ]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.81      0.82      0.81     14669
           1       0.81      0.80      0.81     14568

    accuracy                           0.81     29237
   macro avg       0.81      0.81      0.81     29237
weighted avg       0.81      0.81      0.81     29237



In [50]:
# 파이프라인을 통해서 전처리와 모델 학습을 연결

pipe = Pipeline(
    [
        (
            'vec', CountVectorizer(
                tokenizer=tokenize,
                lowercase=False
            )
        ),
        (
            'model', LinearSVC()
        )
    ]
)

In [51]:
# pipe를 이용해서 매개변수 조합

param_grid = {
    'vec__binary' : [True, False],
    'model__C': [0.9, 1.0]
}

In [52]:
# 교차 검증

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [58]:
grid = GridSearchCV(
    estimator = pipe,
    param_grid = param_grid,
    cv = cv,
    # n_jobs = -1,
    verbose = 1
)

In [59]:
y_1 = y[:20000]
y_1.value_counts()

label
0    10060
1     9940
Name: count, dtype: int64

In [60]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.8, stratify=y, random_state=42
)

In [61]:
X_vali, X_test, y_vali, y_test = train_test_split(
    X_test, y_test, test_size=0.1, stratify=y_test, random_state=42
)

In [62]:
grid.fit(X_train, y_train)

Fitting 3 folds for each of 4 candidates, totalling 12 fits


c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: Use

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...LinearSVC())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.9, 1.0], 'vec__binary': [True, False]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the sco

In [64]:
print('최적의 파라미터:', grid.best_params_)
pred = grid.predict(X_test)
print(classification_report(pred, y_test))

최적의 파라미터: {'model__C': 0.9, 'vec__binary': True}
              precision    recall  f1-score   support

           0       0.81      0.79      0.80      6004
           1       0.78      0.80      0.79      5691

    accuracy                           0.79     11695
   macro avg       0.79      0.79      0.79     11695
weighted avg       0.79      0.79      0.79     11695

